In [ ]:
import numpy as np
import pandas as pd
import napari
import imageio.v2 as imageio
from magicgui import magicgui
from qtpy.QtWidgets import QWidget, QVBoxLayout, QLabel, QTextEdit
from pathlib import Path
import os

In [ ]:
sample_id = 'IMMUcan_2022_WFLOW_10034294-GI-VAR-TIS-UNST-03_004'
sample_name = 'crop_IMMUcan_2022_WFLOW_10034294-GI-VAR-TIS-UNST-03_004'
crop_num = '0000'
dataset = 'IMMUcan'

mask_path = Path(f"/Users/lukashat/sds_mount/sds/sd22c003/phenotyping_benchmark/visual_qc/crop_data/{dataset}/{sample_name}_{crop_num}_mask.tiff") 
annot_path = Path(f"/Users/lukashat/sds_mount/sds/sd22c003/phenotyping_benchmark/datasets/{dataset}/quantification/processed/{dataset}_quantification.csv") 
image_path = Path(f"/Users/lukashat/sds_mount/sds/sd22c003/phenotyping_benchmark/visual_qc/crop_data/{dataset}/{sample_name}_{crop_num}.ome.tiff")
markers_path = Path(f"/Users/lukashat/sds_mount/sds/sd22c003/phenotyping_benchmark/datasets/{dataset}/markers.txt") 


ANNOTATION_COL = "cell_type" 
CELL_ID_COL = "cell_id"  
NEW_ANNOTATION_COL = "NewAnnotation_Lukas"
PROBABILITY_COL = "probability"
REANNOTATED_COL = "Reannotated"
OUTPUT_CSV_PATH = Path(f"/Users/lukashat/sds_mount/sds/sd22c003/phenotyping_benchmark/visual_qc/corrected_annotations/{dataset}/{sample_name}_{crop_num}_reannotated_PA.csv")
os.makedirs(OUTPUT_CSV_PATH.parent, exist_ok=True)

def build_napari_position_aligned_features(
    features: pd.DataFrame,
    seg_labels: np.ndarray,
    cell_id_col: str,
) -> pd.DataFrame:
    """
    Build a features table where row position == label ID.
    This is a workaround for napari setups that ignore DataFrame index values.
    """
    if cell_id_col not in features.columns:
        raise ValueError(f"Missing required column: {cell_id_col}")

    df = features.copy()
    df[cell_id_col] = pd.to_numeric(df[cell_id_col], errors="coerce")
    df = df.dropna(subset=[cell_id_col]).copy()
    df[cell_id_col] = df[cell_id_col].astype(np.int64)

    # keep one row per cell id
    df = df.drop_duplicates(subset=[cell_id_col], keep="first").copy()

    max_id = int(np.max(seg_labels))
    full_index = pd.RangeIndex(0, max_id + 1)

    # record real IDs before reindex so _is_dummy is computed correctly
    original_ids = set(df[cell_id_col].values)

    # align by cell_id, then expand to all label IDs
    aligned = df.set_index(cell_id_col).reindex(full_index)

    # mark dummy rows before adding cell_id (adding cell_id first makes isna().all() always False)
    aligned["_is_dummy"] = ~pd.Index(full_index).isin(original_ids)

    # keep cell id explicit in column (not relying on index in app logic)
    aligned[cell_id_col] = aligned.index.astype(np.int64)

    # defaults for required columns if present
    if ANNOTATION_COL in aligned.columns:
        aligned[ANNOTATION_COL] = aligned[ANNOTATION_COL].fillna("")
    if NEW_ANNOTATION_COL in aligned.columns:
        aligned[NEW_ANNOTATION_COL] = aligned[NEW_ANNOTATION_COL].fillna(pd.NA)
    if REANNOTATED_COL in aligned.columns:
        aligned[REANNOTATED_COL] = aligned[REANNOTATED_COL].fillna(False)

    # optional: probability default for dummy rows
    if PROBABILITY_COL in aligned.columns:
        aligned[PROBABILITY_COL] = pd.to_numeric(aligned[PROBABILITY_COL], errors="coerce").fillna(1.0)

    return aligned.reset_index(drop=True)


class LabelThresholdWidget(QWidget):
    def __init__(
        self,
        viewer: napari.Viewer,
        seg_layer: napari.layers.Labels,
        annotation_col: str = ANNOTATION_COL,
        new_annotation_col: str = NEW_ANNOTATION_COL,
        probability_col: str = PROBABILITY_COL,
        reannotated_col: str = REANNOTATED_COL,
        cell_id_col: str = CELL_ID_COL,
        output_csv_path: Path = OUTPUT_CSV_PATH,
        initial_threshold: float = 0.2,
        all_annotation_choices: list = None,
    ):
        super().__init__()
        self.viewer = viewer
        self.seg_layer = seg_layer

        self.annotation_col = annotation_col
        self.new_annotation_col = new_annotation_col
        self.probability_col = probability_col
        self.reannotated_col = reannotated_col
        self.cell_id_col = cell_id_col
        self.output_csv_path = output_csv_path

        self.current_threshold = initial_threshold
        self.current_annotation = ""
        self.last_selected_id = None

        # full cell-type vocabulary for the reannotate dropdown
        self._all_annotation_choices = all_annotation_choices or self.annotation_choices()

        self.info_label = QLabel("Selected Label Info:")
        self.info_box = QTextEdit()
        self.info_box.setReadOnly(True)

        @magicgui(
            threshold={"widget_type": "FloatSlider", "min": 0.0, "max": 1.0, "step": 0.01},
            auto_call=True,
        )
        def threshold_control(threshold: float = initial_threshold):
            self.current_threshold = float(threshold)
            self.apply_filters()

        @magicgui(
            annotation={"widget_type": "ComboBox", "choices": self.annotation_choices()},
            auto_call=True,
        )
        def annotation_control(annotation: str = ""):
            self.current_annotation = annotation
            self.apply_filters()

        @magicgui(
            call_button="Re-annotate Selected Label",
            new_annotation={"widget_type": "ComboBox", "choices": self._all_annotation_choices},
        )
        
        def reannotate_label(new_annotation: str = ""):
            features = self.get_features()
            if features is None:
                print("No features dataframe found.")
                return
            if self.cell_id_col not in features.columns:
                print(f"Missing required column: {self.cell_id_col}")
                return
            if new_annotation in ("", None):
                print("Please select a valid annotation.")
                return
            if self.last_selected_id is None:
                print("No label selected. Double-click a label first.")
                return

            if self.new_annotation_col not in features.columns:
                features[self.new_annotation_col] = pd.NA
            if self.reannotated_col not in features.columns:
                features[self.reannotated_col] = False

            pos = self._row_positions_for_cell_id(features, self.last_selected_id)
            if pos.size == 0:
                print(f"Cell ID {self.last_selected_id} not found.")
                return

            first_pos = int(pos[0])
            old_val = self.effective_annotation(features.iloc[first_pos])

            new_col_pos = features.columns.get_loc(self.new_annotation_col)
            reann_col_pos = features.columns.get_loc(self.reannotated_col)

            # Update ALL matching rows for this cell_id_col value
            features.iloc[pos, new_col_pos] = new_annotation
            features.iloc[pos, reann_col_pos] = True

            self.seg_layer.features = features
            self.seg_layer.refresh()
            self.apply_filters()
            self._update_info_display(self.last_selected_id)

            # filter dropdown: only types present in this image
            self.annotation_control["annotation"].choices = self.annotation_choices()
            # reannotate dropdown: keep full vocabulary unchanged

            self.save_features_csv()
            print(f"Cell {self.last_selected_id} re-annotated from '{old_val}' to '{new_annotation}'.")

        self.threshold_control = threshold_control
        self.annotation_control = annotation_control
        self.reannotate_label = reannotate_label

        layout = QVBoxLayout()
        layout.addWidget(QLabel("<b>CellDecipher Annotation</b>"))
        layout.addWidget(QLabel("Probability Threshold:"))
        layout.addWidget(self.threshold_control.native)
        layout.addWidget(QLabel("Filter by Annotation:"))
        layout.addWidget(self.annotation_control.native)
        layout.addWidget(QLabel("<b>Re-annotate</b>"))
        layout.addWidget(self.reannotate_label.native)
        layout.addWidget(self.info_label)
        layout.addWidget(self.info_box)
        self.setLayout(layout)

        @self.seg_layer.mouse_double_click_callbacks.append
        def show_label_info(layer, event):
            value = layer.get_value(event.position, world=True)
            if value is None or value == 0:
                self.info_box.setText("No label selected.")
                self.last_selected_id = None
                return

            cell_id = self.resolve_cell_id_from_clicked_label(value)
            if cell_id is None:
                self.info_box.setText(f"Clicked label {value}: no matching row in '{self.cell_id_col}'.")
                self.last_selected_id = None
                return

            self.last_selected_id = cell_id
            self._update_info_display(cell_id)

        self.apply_filters()
        self.viewer.layers.selection.active = self.seg_layer

    def get_features(self):
        return getattr(self.seg_layer, "features", None)

    def _match_cell_id(self, df: pd.DataFrame, cell_id):
        if self.cell_id_col not in df.columns:
            return pd.Series([False] * len(df))
        col_num = pd.to_numeric(df[self.cell_id_col], errors="coerce")
        key_num = pd.to_numeric(pd.Series([cell_id]), errors="coerce").iloc[0]

        if pd.notna(key_num):
            return col_num == key_num

        return df[self.cell_id_col].astype(str) == str(cell_id)

    def _row_positions_for_cell_id(self, df: pd.DataFrame, cell_id) -> np.ndarray:
        if df is None or self.cell_id_col not in df.columns or len(df) == 0:
            return np.array([], dtype=np.int64)

        col = df[self.cell_id_col]
        col_num = pd.to_numeric(col, errors="coerce")
        key_num = pd.to_numeric(pd.Series([cell_id]), errors="coerce").iloc[0]

        if pd.notna(key_num):
            mask = (col_num.to_numpy() == key_num)
        else:
            mask = (col.astype(str).to_numpy() == str(cell_id))

        return np.flatnonzero(mask).astype(np.int64)

    def resolve_cell_id_from_clicked_label(self, clicked_label):
        features = self.get_features()
        if features is None:
            return None

        pos = self._row_positions_for_cell_id(features, clicked_label)
        if pos.size == 0:
            return None

        # Return exact value stored in cell_id_col for that row
        return features[self.cell_id_col].iloc[int(pos[0])]

    def effective_annotation(self, row_or_df):
        if isinstance(row_or_df, pd.Series):
            newv = row_or_df.get(self.new_annotation_col, pd.NA)
            if pd.notna(newv) and str(newv) != "":
                return str(newv)
            return str(row_or_df.get(self.annotation_col, ""))
        else:
            if self.new_annotation_col in row_or_df.columns:
                newv = row_or_df[self.new_annotation_col]
            else:
                newv = pd.Series([pd.NA] * len(row_or_df))
            base = row_or_df[self.annotation_col].astype(str)
            return np.where(pd.notna(newv) & (newv.astype(str) != ""), newv.astype(str), base)

    def annotation_choices(self):
        features = self.get_features()
        if features is None or self.annotation_col not in features.columns:
            return [""]
        eff = self.effective_annotation(features)
        vals = pd.Series(eff).dropna().astype(str).unique().tolist()
        return [""] + sorted(vals)

    def _update_info_display(self, cell_id):
        features = self.get_features()
        if features is None or self.cell_id_col not in features.columns:
            self.info_box.setText("No feature info available.")
            return

        pos = self._row_positions_for_cell_id(features, cell_id)
        if pos.size == 0:
            self.info_box.setText(f"Cell ID {cell_id} not found in features.")
            return

        row = features.iloc[int(pos[0])].to_dict()
        row["EffectiveAnnotation"] = self.effective_annotation(features.iloc[int(pos[0])])

        text = f"<b>Selected cell ID: {cell_id}</b>\n" + "-" * 40 + "\n"
        for k, v in row.items():
            text += f"{k}: {v}\n"

        is_reannotated = bool(row.get(self.reannotated_col, False))
        text += "-" * 40 + "\n"
        text += (
            "<b style='color: orange;'>⚠ Status: REANNOTATED</b>"
            if is_reannotated
            else "<b style='color: green;'>✓ Status: ORIGINAL</b>"
        )
        self.info_box.setText(text)

    def ids_from_features(self, df: pd.DataFrame) -> np.ndarray:
        if df is None or len(df) == 0 or self.cell_id_col not in df.columns:
            return np.array([], dtype=np.int64)
        return pd.to_numeric(df[self.cell_id_col], errors="coerce").dropna().astype(np.int64).to_numpy()

    def apply_filters(self):
        features = self.get_features()
        if features is None:
            return
        required = [self.probability_col, self.annotation_col, self.cell_id_col]
        if any(col not in features.columns for col in required):
            return

        prob = pd.to_numeric(features[self.probability_col], errors="coerce")
        eff = pd.Series(self.effective_annotation(features))

        thresh_mask = prob <= self.current_threshold
        if self.current_annotation not in ("", None):
            ann_mask = eff.astype(str) == str(self.current_annotation)
            selected = features.loc[thresh_mask & ann_mask]
        else:
            selected = features.loc[thresh_mask]

        keep_ids = self.ids_from_features(selected)

        seg = self.seg_layer.data
        seg_ids = np.unique(seg)
        seg_ids = seg_ids[seg_ids != 0]
        keep_ids = np.intersect1d(keep_ids, seg_ids)

        out = np.where(np.isin(seg, keep_ids), seg, 0).astype(seg.dtype)

        if "Filtered" in self.viewer.layers:
            self.viewer.layers["Filtered"].data = out
        else:
            self.viewer.add_labels(out, name="Filtered")
        self.viewer.layers["Filtered"].contour = 1

    def save_features_csv(self):
        features = self.get_features()
        if features is None:
            return
        out = features.copy()
        if "_is_dummy" in out.columns:
            out = out[~out["_is_dummy"].fillna(False)].copy()
        out.to_csv(self.output_csv_path, index=False)
        print(f"Saved: {self.output_csv_path}")


if __name__ == "__main__":

    image = imageio.imread(image_path)
    mask = imageio.imread(mask_path).astype(np.uint32)
    mask_ids = np.unique(mask)
    mask_ids = mask_ids[mask_ids != 0]

    labels_full = pd.read_csv(annot_path)
    # collect all cell types from the complete dataset before any filtering
    all_cell_types = [""] + sorted(
        labels_full[ANNOTATION_COL].dropna().astype(str).unique().tolist()
    )
   
    labels = labels_full[labels_full['sample_id'] == sample_id].copy()
    labels = labels[labels[CELL_ID_COL].isin(mask_ids)]
    labels = labels.set_index(CELL_ID_COL, drop=False)

    if CELL_ID_COL not in labels.columns:
        raise ValueError(f"Required cell ID column '{CELL_ID_COL}' not found in {annot_path}")

    has_probability = PROBABILITY_COL in labels.columns
    if not has_probability:
        labels[PROBABILITY_COL] = 1.0
    if ANNOTATION_COL not in labels.columns:
        labels[ANNOTATION_COL] = "Unknown"
    if NEW_ANNOTATION_COL not in labels.columns:
        labels[NEW_ANNOTATION_COL] = pd.NA
    if REANNOTATED_COL not in labels.columns:
        labels[REANNOTATED_COL] = False

    channel_names = markers_path.read_text().splitlines()
    if len(channel_names) != image.shape[0]:
        print(f"Warning: {len(channel_names)} markers but image has {image.shape[0]} channels — falling back to numeric names.")
        channel_names = [str(i) for i in range(image.shape[0])]

    napari_features = build_napari_position_aligned_features(
        labels, mask, CELL_ID_COL
        )

    viewer = napari.Viewer()
    seg_layer = viewer.add_labels(mask, name="Segmentation", features=napari_features, opacity=0.2)
    viewer.add_image(image, name=channel_names, channel_axis=0, opacity=0.8, visible=False)
    seg_layer.contour = 1

    combined_widget = LabelThresholdWidget(
        viewer,
        seg_layer,
        annotation_col=ANNOTATION_COL,
        new_annotation_col=NEW_ANNOTATION_COL,
        probability_col=PROBABILITY_COL,
        reannotated_col=REANNOTATED_COL,
        cell_id_col=CELL_ID_COL,
        output_csv_path=OUTPUT_CSV_PATH,
        initial_threshold=1.0 if not has_probability else 0.2,
        all_annotation_choices=all_cell_types,
    )
    viewer.window.add_dock_widget(combined_widget, area="right")
    viewer.layers.selection.active = seg_layer
    napari.run()


In [ ]:
out

In [ ]:
labels

In [ ]:
labels['Reannotated'].value_counts()